In [ ]:
# 1. 配置

import json
from pathlib import Path

import numpy as np
import pandas as pd

INDEX_CODE = "000903.SH"
START_DATE = "20160101"
END_DATE = "20251231"


In [ ]:
# 2. 历史中证100成分股

snapshot_dates = ["20151231"]

for year in range(2016, 2026):
    snapshot_dates.extend([
        f"{year}0630",
        f"{year}1231"
    ])

snapshot_dates = sorted(set(snapshot_dates))
historical_members = {}

for date in snapshot_dates:
    members = get_index_stocks(
        INDEX_CODE,
        date=date
    )
    historical_members[date] = members
    print(date, len(members))

historical_stock_pool = sorted(
    set().union(*historical_members.values())
)

print("历史股票池:", len(historical_stock_pool))


In [ ]:
# 3. 后复权行情

price_panel = get_price(
    historical_stock_pool,
    start_date=START_DATE,
    end_date=END_DATE,
    fre_step="1d",
    fields=[
        "close",
        "volume",
        "turnover"
    ],
    skip_paused=False,
    fq="post",
    is_panel=1
)

close_data = price_panel["close"]
volume_data = price_panel["volume"]
turnover_data = price_panel["turnover"]

print("收盘价维度:", close_data.shape)
print("非正收盘价:", (close_data <= 0).sum().sum())


In [ ]:
# 4. 历史成员掩码

membership_mask = pd.DataFrame(
    False,
    index=close_data.index,
    columns=close_data.columns
)

for i, snapshot_date in enumerate(snapshot_dates):
    start = pd.Timestamp(snapshot_date)

    if i < len(snapshot_dates) - 1:
        end = pd.Timestamp(snapshot_dates[i + 1])
        date_mask = (
            (membership_mask.index > start)
            & (membership_mask.index <= end)
        )
    else:
        date_mask = membership_mask.index > start

    members = [
        stock
        for stock in historical_members[snapshot_date]
        if stock in membership_mask.columns
    ]

    membership_mask.loc[date_mask, members] = True

print(membership_mask.sum(axis=1).describe())


In [ ]:
# 5. 月度基本面

month_end_dates = (
    pd.Series(
        close_data.index,
        index=close_data.index
    )
    .groupby(close_data.index.to_period("M"))
    .max()
    .tolist()
)

monthly_fundamental_list = []

for i, trade_date in enumerate(month_end_dates):
    members = membership_mask.columns[
        membership_mask.loc[trade_date]
    ].tolist()

    q = query(
        valuation.symbol,
        valuation.pe_ttm,
        valuation.pb,
        valuation.dividend_rate_12_months,
        profit.weighted_roe
    ).filter(
        valuation.symbol.in_(members)
    )

    monthly_data = get_fundamentals(
        q,
        date=trade_date.strftime("%Y%m%d")
    )

    monthly_data["date"] = trade_date
    monthly_fundamental_list.append(monthly_data)

    if (i + 1) % 12 == 0:
        print(f"基本面进度: {i + 1}/{len(month_end_dates)}")

monthly_fundamentals = pd.concat(
    monthly_fundamental_list,
    ignore_index=True
)

print("基本面行数:", len(monthly_fundamentals))


In [ ]:
# 6. 中证100基准

benchmark_data = get_price(
    INDEX_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
    fre_step="1d",
    fields=["close"],
    skip_paused=False,
    fq="post"
)

benchmark_close = benchmark_data["close"].squeeze()
print("基准数据:", benchmark_close.shape)


In [ ]:
# 7. 导出本地文件

export_dir = Path("csi100_export")
export_dir.mkdir(exist_ok=True)

close_data.to_csv(export_dir / "close.csv")
volume_data.to_csv(export_dir / "volume.csv")
turnover_data.to_csv(export_dir / "turnover.csv")
membership_mask.astype("uint8").to_csv(
    export_dir / "membership.csv"
)
monthly_fundamentals.to_csv(
    export_dir / "fundamentals.csv",
    index=False
)
benchmark_close.rename("close").to_csv(
    export_dir / "benchmark.csv"
)

with open(
    export_dir / "historical_members.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        historical_members,
        file,
        ensure_ascii=False,
        indent=2
    )

print("导出目录:", export_dir)

for file_path in sorted(export_dir.iterdir()):
    print(
        file_path.name,
        f"{file_path.stat().st_size / 1024 / 1024:.2f} MB"
    )
